# Practice run of analysing/testing different models on the UNSW_NB15 dataset, before trying Deep Learning.

Prior research suggests this is a largely non-linear, less separable dataset so deep learning may be necessary, but I will try simpler, more interpretable models first for the sake of completeness, and to gain Variable Importances

In [1]:
#import packages:


from google.colab import drive

try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

if IN_COLAB:
  # Check if drive is mounted by looking for the mount point in the file system.
  # This is a more robust approach than relying on potentially internal variables.
  import os
  if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
  else:
    print("Google Drive is already mounted.")
else:
  print("Not running in Google Colab. Drive mounting skipped.")

from IPython import get_ipython
from IPython.display import display
import cudf
import cuml
from cuml.preprocessing import StandardScaler
from cuml.model_selection import StratifiedKFold, GridSearchCV
from cuml.linear_model import LogisticRegression
import pandas as pd
import sklearn as sk
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

from tqdm import tqdm


print("New run: Packages loaded")

Google Drive is already mounted.
New run: Packages loaded


Let's load our packages and data

In [2]:
#if using colabs - will need to first mount your drive

#change these for different users
test_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_testing-set.parquet'
training_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_training-set.parquet'

# Import the two CSV files
test_set = pd.read_parquet(test_set_filepath)
train_set = pd.read_parquet(training_set_filepath)

print("Data loaded")


Data loaded


The next cell does some basic analysis, and one hot encodes some of the features:

In [4]:


def preprocess_data(data_set):
    # Ensure data_set is a cuDF DataFrame
    if not isinstance(data_set, cudf.DataFrame):
        data_set = cudf.DataFrame.from_pandas(data_set)

    if 'attack_cat' in data_set.columns.to_list():
        data_set = data_set.drop('attack_cat', axis=1)

    if 'proto' in data_set.columns.to_list():
        category_percentages = data_set['proto'].value_counts(normalize=True) * 100
        top_6_categories = category_percentages.head(6).index.to_pandas().tolist()

        # Use 'mask' to replace values not in top_6_categories with 'other'
        data_set['proto_grouped'] = data_set['proto'].mask(
            ~data_set['proto'].isin(top_6_categories), 'other'
        )
        data_set = data_set.drop('proto', axis=1)
        data_set = cudf.get_dummies(data_set, columns=['proto_grouped'], prefix='proto_grouped')

    # No need to drop 'proto_grouped' here since it's already one-hot encoded
    # But if 'proto_grouped' somehow remains, we can drop it
    if 'proto_grouped' in data_set.columns.to_list():
        data_set = data_set.drop(['proto_grouped'], axis=1)

    # One-hot encode any remaining categorical columns
    categorical_cols = data_set.select_dtypes(include=['category']).columns.to_list()
    data_set = cudf.get_dummies(data_set, columns=categorical_cols, prefix_sep='_')

    # Convert boolean columns to integers
    binary_cols = data_set.select_dtypes(include=['bool']).columns
    data_set[binary_cols] = data_set[binary_cols].astype('int32')

    return data_set



train_set = preprocess_data(train_set)
print(f"Data set preprocessed, columns = {len(train_set.columns.to_list())}")

print(f"Data set preprocessed, columns = {train_set.columns.to_list()}")





Data set preprocessed, columns = ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'label', 'proto_grouped_3pc', 'proto_grouped_a/n', 'proto_grouped_aes-sp3-d', 'proto_grouped_any', 'proto_grouped_argus', 'proto_grouped_aris', 'proto_grouped_arp', 'proto_grouped_ax.25', 'proto_grouped_bbn-rcc', 'proto_grouped_bna', 'proto_grouped_br-sat-mon', 'proto_grouped_cbt', 'proto_grouped_cftp', 'proto_grouped_chaos', 'proto_grouped_compaq-peer', 'proto_grouped_cphb', 'proto_grouped_cpnx', 'proto_grouped_crtp', 'proto_grouped_crudp', 'proto_grouped_dcn', 'proto_grouped_ddp', 'proto_grouped_ddx', 'proto_grouped_dgp', 'proto_grouped_egp', 'proto_grouped_eigrp', 'proto_grouped_emcon', 'proto_grouped_en

NOTE TO SELF -
1. THIS IS FOR BINARY CLASSIFICATION, WE WANT MULTICLASS EVENTUALLY, BUT FOR NOW WE WILL JUST DO BN


Based on the high number of columns in the Proto column, we may want to consider an Embeddings layer with the Deep Learning that we plan to undertake later. However since DT/RF perform somewhat poorly on sparse vector datasets (like one hot encoded ones) we will group all the extremely rare categories into an 'other'.


In [5]:
def run_models(model_type, test_set = test_set, train_set = train_set):
  """
  Runs LR, DT or RF model on dataframe
  """

  #drop label and define list of out targets
  X_train = train_set.drop('label', axis=1)
  y_train = train_set['label']

  #do the same for test set
  X_test = test_set.drop('label', axis=1)
  y_test = test_set['label']

  # List only the categorical columns (object types)
  categorical_cols = train_set.select_dtypes(include=['category']).columns.tolist()


#=============================== LR ========================================#

  if model_type.upper() == 'LR':

    # Define the pipeline
    pipeline = Pipeline([
          ('scaler', StandardScaler()),
          ('classifier', LogisticRegression(max_iter=1000))
      ])

    # Define the hyperparameter grid
    param_grid = {
        'classifier__C': [0.01, 0.1, 1, 10, 100],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__solver': ['liblinear']  # Suitable for smaller datasets
    }

    # Set up cross-validation strategies
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    print("KFolds defined, beginning nested cross-validation.")

    # Outer loop (on training data)
    outer_scores = []

    #outer loop
    for train_index, val_index in tqdm(outer_cv.split(X_train, y_train)):
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        # Set up GridSearchCV with the pipeline (inner loop)
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring='roc_auc',  # Use appropriate scoring for your task
        )

        # Fit GridSearchCV on the training fold
        grid_search.fit(X_train_fold, y_train_fold)

        # Evaluate on the validation fold
        score = grid_search.score(X_val_fold, y_val_fold)
        outer_scores.append(score)

    # Print the average outer score (ROC AUC) on training data
    print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

    # Evaluate the best model on the held-out test set
    best_model = grid_search.best_estimator_
    test_score = best_model.score(X_test, y_test)
    print(f"Test ROC AUC: {test_score}")

#=============================== DT ========================================#

  if model_type.upper() == 'DT':

    param_grid_dt = {
    'max_depth': [3, 5, 10],             # List of max depth values
    'min_samples_split': [2, 10, 20],    # List of minimum samples to split
    'min_samples_leaf': [1, 5, 10]       # List of minimum samples per leaf
    }

    pass
#=============================== RF ========================================#

  if model_type.upper() == 'RF':

    param_grid_rf = {
    'n_estimators': [50, 100, 200],      # List of number of trees to use
    'max_depth': [5, 10, 20],            # List of maximum depths for trees
    'min_samples_split': [2, 10, 20]     # List of minimum samples for splitting a node
    }

    pass

run_models('LR')

KFolds defined, beginning nested cross-validation.


0it [00:00, ?it/s]


TypeError: StratifiedKFold.get_n_splits() got an unexpected keyword argument 'groups'